# HerWILL Datathon: production ensemble inference

Loads the **`ensemble_bundle/`** written by `ensemble_tuning.ipynb` and the six `{tag}_final/`
checkpoints, then predicts. **Nothing is fitted here:** temperatures, blend weights, class weights,
kNN settings and both memories all come from the bundle.

**Inputs (attach as Kaggle datasets):**
- `ensemble_bundle/`: `ensemble_config.json` (plus `lookup.json` / `knn_index.joblib` only if the
  config switches on exact-match / kNN; the equal-weight bundle needs just the config)
- the checkpoint folders `hatebert_final/`, `twitter_xlmr_final/`, `xlmr_large_final/`,
  `banglabert_final/`, `banglishbert_final/`, `banglabert_large_final/`
- the text file to label (`test.csv`, with columns `id, text`)

**Offline:** models load from local folders, so no internet is needed, except to `pip install` the
csebuetnlp normalizer, which installs from the `wheels/` folder of the `herwill-offline-assets` dataset when it is attached.

**Output:** `submission.csv` (`id, y_pred` and optionally `parameters`), `test_probs.npy`,
`test_predictions_detailed.csv`.

**This notebook must run the bundle exactly as written.** Do not drop members by hand: an assertion below checks that every member named in `ensemble_config.json` resolves to a checkpoint, and section 6b prints which post-processing stages are actually live.

## 1. Config

In [ ]:
BUNDLE_DIR = "/kaggle/input/datasets/supriopaul844/config-ensemble"     # folder containing ensemble_config.json (auto-searched if None)
TEST_PATH  = "/kaggle/input/datasets/supriopaul844/up-test/test.csv"      # the competition test set to label
CKPT_ROOTS = CKPT_ROOTS = [
    "/kaggle/input/models/supriopaul844/bangla-finetune-up/pytorch/default/1",
    "/kaggle/input/models/supriopaul844/english-finetuned-up/pytorch/default/1", ]


EVAL_BS = 256
ADD_PARAMETERS_COLUMN = True     # HerWILL submission format: id, y_pred, parameters

## 2. Environment

In [ ]:
import os, re, gc, json, time, random, warnings, subprocess, sys
from pathlib import Path

# Everything loads from local checkpoint folders; never touch the HF Hub.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel, AutoConfig
import transformers

warnings.filterwarnings("ignore")

N_CLASSES = 3
CLASS_NAMES = {0: "Explicitly Toxic", 1: "Subtly Toxic", 2: "Non-toxic"}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    AMP_DTYPE = None

OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("artifacts")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("transformers", transformers.__version__, "| torch", torch.__version__)
print("device", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "", "| amp", AMP_DTYPE)

## 3. Preprocessing (identical to training)

In [ ]:
# Preprocessing: must be byte-for-byte what the training notebooks did.
#   "multilingual" family (hatebert / twitter_xlmr / xlmr_large) -> _base_clean only
#   "bangla" family (banglabert / banglishbert / banglabert_large) -> csebuetnlp normalizer, then _base_clean
try:
    from normalizer import normalize as bn_normalize
except ImportError:
    # offline: wheel from the offline-assets dataset; online fallback: GitHub
    whl = next(iter(sorted(Path("/kaggle/input").rglob("normalizer-*.whl"))), None) \
        if Path("/kaggle/input").exists() else None
    if whl is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index",
                        "--find-links", str(whl.parent), "normalizer"], check=False)
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "git+https://github.com/csebuetnlp/normalizer"], check=False)
    try:
        from normalizer import normalize as bn_normalize
    except ImportError:
        bn_normalize = None
BN_NORMALIZER_AVAILABLE = bn_normalize is not None
print("csebuetnlp normalizer available:", BN_NORMALIZER_AVAILABLE)

import unicodedata as ud

ZW_RE      = re.compile(r"[​‎‏﻿­⁠]")
URL_RE     = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"(?<!\w)@\w+")
ELONG_RE   = re.compile(r"([A-Za-z])\1{2,}")
PUNCT_RE   = re.compile(r"([!?.,\-*])\1{2,}")
MAX_CHARS  = 2000


def _base_clean(s):
    s = ud.normalize("NFC", s)
    s = ZW_RE.sub("", s)
    s = URL_RE.sub(" [URL] ", s)
    s = MENTION_RE.sub(" [USER] ", s)
    s = ELONG_RE.sub(r"\1\1", s)
    s = PUNCT_RE.sub(r"\1\1", s)
    s = s[:MAX_CHARS]
    s = re.sub(r"\s+", " ", s).strip()
    return s if s else "[EMPTY]"


def clean_multilingual(s):
    return _base_clean(str(s))


def clean_bangla(s):
    s = str(s)
    if bn_normalize is not None:
        try:
            s = bn_normalize(s)
        except Exception:
            pass
    return _base_clean(s)


PREPROCESSORS = {"multilingual": clean_multilingual, "bangla": clean_bangla}


def norm_key(s):
    # Key for the exact-match lookup and the kNN memory. It does not depend on the Bangla
    # normalizer, so it behaves the same whether or not that package is installed.
    return clean_multilingual(s).lower()


def load_csv(path, need_labels):
    df = pd.read_csv(path)
    text_col = "text" if "text" in df.columns else next(
        c for c in df.columns if c.lower() in ("text", "sentence", "comment", "content"))
    label_col = "y" if "y" in df.columns else next(
        (c for c in df.columns if c.lower() in ("y", "label", "labels", "target", "class")), None)
    if need_labels and label_col is None:
        raise ValueError(f"{path} has no label column")
    out = pd.DataFrame({
        "id": df["id"] if "id" in df.columns else np.arange(len(df)),
        "text": df[text_col].fillna("").astype(str),
    })
    if label_col is not None and need_labels:
        out["y"] = df[label_col]
        out = out.dropna(subset=["y"])
        out["y"] = out["y"].astype(int)
    print(f"{Path(path).name}: {len(out)} rows")
    return out.reset_index(drop=True)

## 4. Model loading & inference

In [ ]:
# Same architecture as the training notebooks, rebuilt from the LOCAL checkpoint folder
# (config.json + tokenizer + head_config.json + model_state.pt), so no internet is needed.
class BigHeadClassifier(nn.Module):
    def __init__(self, ckpt_dir):
        super().__init__()
        meta = json.load(open(Path(ckpt_dir) / "head_config.json"))
        config = AutoConfig.from_pretrained(ckpt_dir)
        extra = {"add_pooling_layer": False} if config.model_type in ("bert", "roberta", "xlm-roberta") else {}
        self.encoder = AutoModel.from_config(config, **extra)
        H = config.hidden_size
        layers, d_in = [], 2 * H
        for d_out in meta["hidden_sizes"]:
            layers += [nn.Linear(d_in, d_out), nn.LayerNorm(d_out), nn.GELU()]
            d_in = d_out
        layers.append(nn.Linear(d_in, meta["n_classes"]))
        self.head = nn.Sequential(*layers)
        self.meta = meta

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kw = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kw["token_type_ids"] = token_type_ids
        h = self.encoder(**kw).last_hidden_state
        cls = h[:, 0]
        m = attention_mask.unsqueeze(-1).to(h.dtype)
        mean = (h * m).sum(1) / m.sum(1).clamp(min=1.0)
        return self.head(torch.cat([cls, mean], dim=-1).float())


def find_checkpoint(tag, roots):
    for r in roots:
        r = Path(r)
        if not r.exists():
            continue
        for hc in r.rglob("head_config.json"):
            if hc.parent.name == f"{tag}_final":
                return hc.parent
    return None


def load_member(ckpt_dir):
    model = BigHeadClassifier(ckpt_dir)
    sd = torch.load(Path(ckpt_dir) / "model_state.pt", map_location="cpu", weights_only=True)
    missing, unexpected = model.load_state_dict(sd, strict=False)
    # position_ids buffers differ between transformers versions; anything else is a real mismatch
    missing = [k for k in missing if not k.endswith("position_ids")]
    unexpected = [k for k in unexpected if not k.endswith("position_ids")]
    if missing or unexpected:
        raise RuntimeError(f"state_dict mismatch in {ckpt_dir}: missing={missing[:5]} unexpected={unexpected[:5]}")
    tok = AutoTokenizer.from_pretrained(ckpt_dir)
    return model.to(DEVICE).eval(), tok


@torch.no_grad()
def predict_logits(model, tok, texts, max_len, bs=256):
    enc = tok(list(texts), truncation=True, max_length=max_len)
    keys = list(enc.keys())
    order = np.argsort([len(x) for x in enc["input_ids"]], kind="stable")   # length-sorted = little padding
    out = np.zeros((len(texts), N_CLASSES), dtype=np.float32)
    for s in range(0, len(order), bs):
        sel = order[s:s + bs]
        batch = tok.pad([{k: enc[k][i] for k in keys} for i in sel], return_tensors="pt")
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
        with torch.autocast("cuda", dtype=AMP_DTYPE or torch.float32, enabled=AMP_DTYPE is not None):
            z = model(**batch)
        out[sel] = z.float().cpu().numpy()
    return out


_clean_cache = {}

def run_member(tag, family, ckpt_dir, raw_texts, bs=256):
    ck = (family, id(raw_texts))
    if ck not in _clean_cache:
        _clean_cache[ck] = [PREPROCESSORS[family](t) for t in raw_texts]
    t0 = time.time()
    model, tok = load_member(ckpt_dir)
    n_params = sum(p.numel() for p in model.parameters())
    logits = predict_logits(model, tok, _clean_cache[ck], model.meta["max_len"], bs)
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print(f"  {tag:<18} {n_params/1e6:7.1f}M params  {len(raw_texts)} rows  {time.time()-t0:.0f}s")
    return logits, int(n_params)

## 5. Post-processing (`final_predict`, the same code that was tuned)

In [ ]:
# Post-processing: the ONE function that turns member logits into labels (final_predict).
# Tuning calls it with candidate params; production calls it with the saved params.
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib


def softmax_T(z, T):
    z = z / T
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def macro_f1(y, pred):
    cm = np.bincount(y * N_CLASSES + pred, minlength=N_CLASSES ** 2).reshape(N_CLASSES, N_CLASSES)
    tp = np.diag(cm).astype(float)
    prec = np.divide(tp, cm.sum(0), out=np.zeros(N_CLASSES), where=cm.sum(0) > 0)
    rec = np.divide(tp, cm.sum(1), out=np.zeros(N_CLASSES), where=cm.sum(1) > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros(N_CLASSES), where=(prec + rec) > 0)
    present = (cm.sum(0) + cm.sum(1)) > 0
    return float(f1[present].mean())


def build_lookup(keys, ys):
    # exact-text memory: majority label per normalised text, ties dropped (from exploration.ipynb)
    df = pd.DataFrame({"key": list(keys), "y": list(ys)})
    out = {}
    for key, g in df.groupby("key", sort=False):
        vc = g["y"].value_counts()
        if len(vc) > 1 and vc.iloc[0] == vc.iloc[1]:
            continue
        out[key] = int(vc.index[0])
    return out


def exact_labels(keys, lookup):
    return np.array([lookup.get(k, -1) for k in keys], dtype=np.int64)


class KnnIndex:
    """Char n-gram TF-IDF memory of labelled texts: cosine nearest neighbours of each query."""

    def __init__(self, ngram_range=(2, 5), min_df=2, max_features=500_000):
        self.vec = TfidfVectorizer(analyzer="char_wb", ngram_range=ngram_range, min_df=min_df,
                                   max_features=max_features, sublinear_tf=True,
                                   lowercase=False, dtype=np.float32)
        self.MT, self.labels = None, None

    def fit(self, keys, labels):
        self.MT = self.vec.fit_transform(list(keys)).T.tocsr()
        self.labels = np.asarray(labels, dtype=np.int64)
        return self

    def topk(self, keys, k, chunk=512):
        Q = self.vec.transform(list(keys))
        k = min(k, len(self.labels))
        sims = np.zeros((Q.shape[0], k), dtype=np.float32)
        idx = np.zeros((Q.shape[0], k), dtype=np.int64)
        for s in range(0, Q.shape[0], chunk):
            S = (Q[s:s + chunk] @ self.MT).toarray()
            part = np.argpartition(-S, k - 1, axis=1)[:, :k]
            ps = np.take_along_axis(S, part, 1)
            o = np.argsort(-ps, axis=1)
            idx[s:s + chunk] = np.take_along_axis(part, o, 1)
            sims[s:s + chunk] = np.take_along_axis(ps, o, 1)
        return sims, self.labels[idx]

    def save(self, path):
        joblib.dump({"vec": self.vec, "MT": self.MT, "labels": self.labels}, path, compress=3)

    @classmethod
    def load(cls, path):
        d = joblib.load(path)
        obj = cls.__new__(cls)
        obj.vec, obj.MT, obj.labels = d["vec"], d["MT"], d["labels"]
        return obj


def knn_probs(sims, labels, min_sim):
    w = np.where(sims >= min_sim, sims, 0.0)
    pk = np.stack([(w * (labels == c)).sum(1) for c in range(N_CLASSES)], axis=1)
    has = pk.sum(1) > 0
    pk[has] /= pk[has].sum(1, keepdims=True)
    return pk, has


def final_predict(member_logits, knn_sims, knn_labels, exact, params):
    """member probs (temperature) -> weighted blend -> kNN soft mix -> class weights -> argmax
       -> kNN hard override (near-duplicate) -> exact-match override."""
    mem = params["members"]
    w = np.array([m["weight"] for m in mem], dtype=np.float64)
    w = w / w.sum()
    p = sum(wi * softmax_T(member_logits[m["tag"]], m["temperature"]) for wi, m in zip(w, mem))

    kn = params["knn"]
    if kn["alpha"] > 0:
        pk, has = knn_probs(knn_sims[:, :kn["k"]], knn_labels[:, :kn["k"]], kn["min_sim"])
        p = np.where(has[:, None], (1 - kn["alpha"]) * p + kn["alpha"] * pk, p)

    p = p * np.asarray(params["class_weights"], dtype=np.float64)
    p = p / p.sum(1, keepdims=True)
    pred = p.argmax(1)
    source = np.full(len(pred), "model", dtype=object)

    if kn["tau_hard"] is not None:
        m = knn_sims[:, 0] >= kn["tau_hard"]
        pred[m] = knn_labels[m, 0]; source[m] = "knn_hard"
    if params["exact_match"]:
        m = exact >= 0
        pred[m] = exact[m]; source[m] = "exact"
    return pred, p, source

## 6. Load bundle, checkpoints and input

In [ ]:
def find_one(explicit, pattern):
    if explicit is not None:
        return Path(explicit)
    for r in ["/kaggle/input", "/kaggle/working", "."]:
        if Path(r).exists():
            hits = sorted(Path(r).rglob(pattern))
            if hits:
                return hits[0]
    raise FileNotFoundError(pattern)


cfg_path = find_one(None if BUNDLE_DIR is None else Path(BUNDLE_DIR) / "ensemble_config.json",
                    "ensemble_config.json")
BUNDLE = cfg_path.parent
CFG_B = json.load(open(cfg_path))
PARAMS = CFG_B["params"]
print("bundle:", BUNDLE, "| created", CFG_B.get("created"), "| scores", CFG_B.get("scores"))

if CFG_B["uses_bangla_family"] and CFG_B["bn_normalizer_used"] and not BN_NORMALIZER_AVAILABLE:
    raise RuntimeError("The bundle was tuned WITH the csebuetnlp normalizer but it is not installed here; "
                       "Bangla-family predictions would not match. Install it (or attach its wheel).")

CKPTS = {}
for m in PARAMS["members"]:
    ck = find_checkpoint(m["tag"], CKPT_ROOTS)
    if ck is None:
        raise FileNotFoundError(f"checkpoint {m['tag']}_final/ not found under {CKPT_ROOTS}")
    CKPTS[m["tag"]] = ck
    print(f"  {m['tag']:<18} {ck}")

# The ensemble is only the ensemble if every member the bundle names is actually present.
# Dropping one by hand changes the prediction silently -- so fail loudly instead.
assert len(CKPTS) == len(PARAMS["members"]), (
    f"bundle names {len(PARAMS['members'])} members but {len(CKPTS)} checkpoints resolved")

# The memories are only needed if the bundle switches on exact-match / kNN stages.
NEED_LOOKUP = bool(PARAMS["exact_match"])
NEED_KNN = PARAMS["knn"]["alpha"] > 0 or PARAMS["knn"]["tau_hard"] is not None
LOOKUP = json.load(open(BUNDLE / CFG_B["lookup_file"], encoding="utf-8")) if NEED_LOOKUP else {}
KNN = KnnIndex.load(BUNDLE / CFG_B["knn_file"]) if NEED_KNN else None
print(f"exact-match memory: {len(LOOKUP) if NEED_LOOKUP else 'off'} | "
      f"kNN memory: {len(KNN.labels) if NEED_KNN else 'off'}")

test = load_csv(find_one(TEST_PATH, "test.csv"), need_labels=False)

## 6b. Active configuration

In [ ]:
# What is actually switched on. Print this before predicting -- the bundle is the single
# source of truth, and a stage that is off here contributed nothing to the submission.
w = np.array([m["weight"] for m in PARAMS["members"]], dtype=float)
w = w / w.sum()
kn, cw = PARAMS["knn"], np.asarray(PARAMS["class_weights"], dtype=float)

print(f"members ({len(PARAMS['members'])}):")
for m, wi in zip(PARAMS["members"], w):
    print(f"   {m['tag']:<18} family={m['family']:<12} "
          f"T={m['temperature']:<5} weight={wi:.4f}")

print("\npost-processing stages:")
print(f"   temperature scaling   {'ON' if any(m['temperature'] != 1.0 for m in PARAMS['members']) else 'off (all T=1)'}")
print(f"   blend weights         {'fitted' if len(set(np.round(w, 6))) > 1 else 'EQUAL (fitted weights did not beat equal)'}")
print(f"   kNN soft mix          {'ON alpha=' + str(kn['alpha']) if kn['alpha'] > 0 else 'off'}")
print(f"   class priors          {'ON ' + str(cw.tolist()) if not np.allclose(cw, 1.0) else 'off (all 1.0)'}")
print(f"   kNN hard override     {'ON tau=' + str(kn['tau_hard']) if kn['tau_hard'] is not None else 'off'}")
print(f"   exact-match override  {'ON' if PARAMS['exact_match'] else 'off'}")

if np.allclose(cw, 1.0):
    print("\n*** class priors are all 1.0 -- this bundle was NOT tuned. The submitted run "
          "used fitted priors; check BUNDLE_DIR points at the tuned bundle. ***")

## 7. Predict

In [ ]:
t0 = time.time()
raw = test["text"].tolist()
TEST_LOGITS, N_PARAMS = {}, {}
for m in PARAMS["members"]:
    TEST_LOGITS[m["tag"]], N_PARAMS[m["tag"]] = run_member(m["tag"], m["family"], CKPTS[m["tag"]], raw, EVAL_BS)
TOTAL_PARAMS = int(sum(N_PARAMS.values()))          # counted from the loaded models
print(f"total ensemble parameters: {TOTAL_PARAMS/1e6:.1f}M")

keys = test["text"].map(norm_key).tolist()
exact = exact_labels(keys, LOOKUP)
if NEED_KNN:
    sims, nlab = KNN.topk(keys, CFG_B["knn_k_max"])
else:                                               # placeholders; unused when kNN stages are off
    sims = np.zeros((len(keys), 1), dtype=np.float32)
    nlab = np.full((len(keys), 1), -1, dtype=np.int64)

pred, probs, source = final_predict(TEST_LOGITS, sims, nlab, exact, PARAMS)
print(f"done in {time.time()-t0:.0f}s")
print("prediction source :", pd.Series(source).value_counts().to_dict())
print("label distribution:", pd.Series(pred).value_counts().sort_index().rename(CLASS_NAMES).to_dict())

## 8. Write outputs

In [ ]:
submission = pd.DataFrame({"id": test["id"].values, "y_pred": pred.astype(int)})
if ADD_PARAMETERS_COLUMN:
    submission["parameters"] = TOTAL_PARAMS
assert submission["y_pred"].isin([0, 1, 2]).all()
submission.to_csv(OUT_DIR / "submission.csv", index=False)

np.save(OUT_DIR / "test_probs.npy", probs)
pd.DataFrame({"id": test["id"].values, "y_pred": pred, "source": source,
              "knn_top1_sim": sims[:, 0], "knn_top1_label": nlab[:, 0],
              **{f"p{i}": probs[:, i] for i in range(N_CLASSES)},
              **{f"{t}_argmax": z.argmax(1) for t, z in TEST_LOGITS.items()}}
             ).to_csv(OUT_DIR / "test_predictions_detailed.csv", index=False)

print(submission.head())
print("wrote", OUT_DIR / "submission.csv")

In [ ]:
# ── OPTIONAL: score a submission against a labelled file ──────────────────────
#
# WARNING. Only meaningful if TRUTH was NOT part of training. The old labelled test set
# was released for training after the leak, so scoring against it measures memorisation,
# not generalisation -- it will read ~0.90 and mean nothing. Leave TRUTH = None unless you
# have a genuinely held-out labelled file.
TRUTH = None          # e.g. "/kaggle/input/.../held_out_labeled.csv"
SUB   = OUT_DIR / "submission.csv"

if TRUTH is None:
    print("scoring skipped (TRUTH is None) -- the leaderboard is the only honest score here")
else:
    from sklearn.metrics import f1_score, classification_report, confusion_matrix

    def _col(df, aliases, what):
        low = {c.lower(): c for c in df.columns}
        for a in aliases:
            if a in low:
                return low[a]
        raise KeyError(f"no {what} column in {list(df.columns)}; looked for {aliases}")

    sub_df = pd.read_csv(SUB)
    truth  = pd.read_csv(TRUTH)
    sc = _col(sub_df, ["y_pred", "prediction", "pred", "label", "y", "target"], "prediction")
    tc = _col(truth,  ["y", "label", "y_true", "target", "class"], "truth")

    m = (truth[[_col(truth, ["id"], "id"), tc]].rename(columns={tc: "y_true"})
         .merge(sub_df[[_col(sub_df, ["id"], "id"), sc]].rename(columns={sc: "y_pred"}),
                on="id", how="left"))

    assert len(m) == len(truth), f"{len(m)} joined vs {len(truth)} truth rows"
    assert m.y_pred.notna().all(), f"{m.y_pred.isna().sum()} ids missing from the submission"
    assert set(m.y_pred.unique()) <= {0, 1, 2}, f"bad labels: {sorted(set(m.y_pred.unique()))}"

    yt, yp = m.y_true.values.astype(int), m.y_pred.values.astype(int)
    print(f"MACRO F1 : {f1_score(yt, yp, average='macro'):.5f}    <- competition metric")
    print(f"accuracy : {(yt == yp).mean():.5f}    rows: {len(m)}\n")
    print(classification_report(yt, yp, digits=3, zero_division=0,
                                target_names=[CLASS_NAMES[i] for i in range(N_CLASSES)]))
    cm = confusion_matrix(yt, yp, labels=[0, 1, 2])
    print("confusion matrix (rows = true):\n", cm)
    n_err = cm.sum() - np.trace(cm)
    if n_err:
        print(f"0<->1 confusion: {cm[0,1]+cm[1,0]} of {n_err} errors "
              f"({(cm[0,1]+cm[1,0])/n_err:.1%})")
    print("predicted mix:", pd.Series(yp).value_counts(normalize=True).sort_index().round(4).to_dict())
    print("true mix     :", pd.Series(yt).value_counts(normalize=True).sort_index().round(4).to_dict())